In [1]:
%cd ../..

/run/media/nazif/2F946E411BA61D49/mirscribe


In [2]:
import pandas as pd
import concurrent.futures
from scripts.utils import *
from scripts.ensembl import *

g37 = import_pyensembl(37)

INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.pep.all.fa.gz.pickle


In [ ]:
def get_transcript_id(coord):
    if transcript_id := g37.transcript_ids_at_locus(*coord):
        return tuple(coord), transcript_id
    else:
        return tuple(coord), "not_found"

In [ ]:
relevant_cols = ["id", "chr", "start_coordinate", "end_coordinate", "mirna_accession", "is_mutated", "prediction", "binary_prediction", "pred_difference", "pred_difference_binary"]
filename = "results/sana_results_0_1500_with_prediction_only_meaningful_results"

df = pd.read_csv(f"{filename}.csv", low_memory=False, usecols=relevant_cols)
len(df) / df.id.nunique()

### explanation
there are 2 entries for every id. One for wild type and one for mutated. Which is encoded in "is_mutated" column.


In [ ]:
# /2 is there because there are 2 entries for every id.
df.pred_difference_binary.value_counts() / 2

# pyensembl

In [ ]:
# adding ENST IDs

coords = df[['chr', 'start_coordinate']].values.tolist()

with concurrent.futures.ProcessPoolExecutor() as executor:
    results = executor.map(get_transcript_id, coords)
    
ensts = dict(results)
df["ENST"] = [ensts.get((row["chr"], row["start_coordinate"]), "") for _, row in df.iterrows()]

# report
print(f"{len(df[df.ENST == 'not_found']) / len(df):.3%} of the coordinates have no transcripts")


In [ ]:
df.to_csv(f"{filename}_with_ENSTs.csv")

In [ ]:
df[df.is_mutated == 0]

In [ ]:
cols = ["id", "mirna_accession", "ENST", "is_mutated", "pred_difference_binary"]


gain_df = df[cols][(df[cols].is_mutated == 0) & (df[cols].pred_difference_binary == 1)] 
loss_df = df[cols][(df[cols].is_mutated == 0) & (df[cols].pred_difference_binary == -1)] 

gain_df.drop(columns=["is_mutated", "pred_difference_binary"], inplace=True)
loss_df.drop(columns=["is_mutated", "pred_difference_binary"], inplace=True)

# remove the last 13 characters (MIMAT0000070 like identifiers)
gain_df['id'] = gain_df['id'].str[:-13]
loss_df['id'] = loss_df['id'].str[:-13]


In [ ]:
gain_df.to_csv("results/gain_pairs.csv", index=False)

In [ ]:
loss_df.to_csv("results/loss_pairs.csv", index=False)
